# Final Retraining Pipeline

Trains the **best configuration found by each candidate algorithm** (GA, SADE,
PSO, GWO, BO) for the categorical 7-HP search space of Study 1, under a full
training budget, on **both datasets**. Each configuration is trained with 3
initialization seeds to estimate the variance of the retraining.

**Protocol**
- 2 datasets x 5 algorithms x 3 seeds = 30 retrainings
- 30 epochs, EarlyStopping(patience=5) on `val_accuracy`, restore best weights
- per-epoch history (`accuracy`, `val_accuracy`, `loss`, `val_loss`) recorded
  for every retraining

The configurations of a dataset are always selected from the searches run on
**that** dataset. This used to be set separately from the dataset being
trained, which is how an earlier CIFAR-10 retraining ended up training the
MNIST-selected configurations.

**Outputs**, written under `code/colab/results/HPO/retrain/<dataset>/`
- `retrain_results_bo.json` - full structured results
- `retrain_val_accuracy_per_epoch.{png,pdf}` and `retrain_val_loss_per_epoch.{png,pdf}`
- `retrain_summary.{csv,md}` and `retrain_configs.csv`


## 1. Imports and configuration


In [ ]:
import os, sys, json, time, random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
from keras.callbacks import EarlyStopping

# allow imports from sibling utils/
ROOT = Path.cwd()
if (ROOT / 'utils').exists():
    sys.path.insert(0, str(ROOT))
elif (ROOT.parent / 'utils').exists():
    sys.path.insert(0, str(ROOT.parent))

from utils.datasets import getDataset
from utils.models   import buildTemplateCNN

print('TF :', tf.__version__)
print('Keras:', keras.__version__)


ModuleNotFoundError: No module named 'tensorflow'

: 

In [ ]:
# --- what to run -----------------------------------------------------------
DATASETS   = ['mnist', 'cifar10']
ALGORITHMS = ['ga', 'sade', 'pso', 'gwo', 'bo']

SEEDS    = [0, 1, 2]
EPOCHS   = 30
PATIENCE = 5

# --- where the Study 1 search runs live ------------------------------------
RESULTS_ROOT = ROOT / 'colab' / 'results' / 'HPO'
if not RESULTS_ROOT.exists():
    RESULTS_ROOT = ROOT.parent / 'code' / 'colab' / 'results' / 'HPO'
if not RESULTS_ROOT.exists():
    raise FileNotFoundError(f'HPO results not found, looked under {RESULTS_ROOT}')

# the run folders are not named consistently: bo/cifar10 against ga/cifar-10
DS_ALIASES = {'mnist': ['mnist'], 'cifar10': ['cifar10', 'cifar-10']}

print('results root :', RESULTS_ROOT)
print('datasets     :', DATASETS)
print('retrainings  :', len(DATASETS) * len(ALGORITHMS) * len(SEEDS))


## 2. Locate the best-of-5 config per algorithm, for each dataset

For every algorithm and every dataset, the search runs of *that* dataset
are scanned and the one with the highest `best.val_accuracy` is picked.
Its configuration is what will be retrained.


In [ ]:
def resolve_run_dir(algo: str, dataset: str) -> Path:
    """The folder holding `algo`'s search runs for `dataset`."""
    for alias in DS_ALIASES[dataset]:
        d = RESULTS_ROOT / algo / alias
        if d.is_dir() and any(d.glob('*.json')):
            return d
    tried = ', '.join(str(RESULTS_ROOT / algo / a) for a in DS_ALIASES[dataset])
    raise FileNotFoundError(f'no search runs for {algo} on {dataset}; tried {tried}')


def find_best_config(algo: str, dataset: str) -> dict:
    """The configuration of the highest-val_accuracy run, which is what gets
    retrained. Fails loudly: returning None here is what allowed a silent
    fallback to the wrong dataset before."""
    run_dir = resolve_run_dir(algo, dataset)
    best = None
    for jp in sorted(run_dir.glob('*.json')):
        with open(jp, 'r') as f:
            d = json.load(f)
        if best is None or d['best']['val_accuracy'] > best['val_accuracy']:
            best = {
                'algorithm':    d.get('algorithm', algo.upper()),
                'tag':          jp.stem,
                'dataset':      dataset,
                'run_dir':      str(run_dir),
                'val_accuracy': d['best']['val_accuracy'],
                'fitness':      d['best']['fitness'],
                'config':       d['best']['config'],
            }
    if best is None:
        raise RuntimeError(f'{algo}/{dataset}: no usable run in {run_dir}')
    return best


def best_configs_for(dataset: str) -> dict:
    out = {}
    print(f'=== {dataset} ===')
    for algo in ALGORITHMS:
        info = find_best_config(algo, dataset)
        out[algo.upper()] = info
        cfg = info['config']
        print(f'  {algo.upper():<5} {info["tag"]:<26} val_acc={info["val_accuracy"]:.4f}  '
              f'bs={cfg["batch_size"]}, f0={cfg["base_filters"]}, blocks={cfg["n_blocks"]}, '
              f'dense={cfg["dense_units"]}, opt={cfg["optimizer"]}, pool={cfg["pooling"]}, '
              f'k={cfg["kernel_size"]}')
    return out


# a quick look before committing hours of GPU time: the two datasets must not
# select the same configurations
selected = {ds: best_configs_for(ds) for ds in DATASETS}
if len(DATASETS) == 2:
    a, b = (selected[d] for d in DATASETS)
    same = [k for k in a if a[k]['config'] == b[k]['config']]
    print(f'\nidentical configurations across the two datasets: {same if same else "none"}')


## 3. Retrain each best config with 3 seeds


In [ ]:
def set_global_seed(seed: int):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    try:
        keras.utils.set_random_seed(seed)
    except AttributeError:
        pass


def train_one(config: dict, seed: int, data: dict):
    set_global_seed(seed)
    input_shape = data['x_train'].shape[1:]
    num_classes = data['num_classes']
    model = buildTemplateCNN(config, input_shape, num_classes)
    es = EarlyStopping(monitor='val_accuracy', patience=PATIENCE,
                       restore_best_weights=True, verbose=0)
    t0 = time.time()
    history = model.fit(
        data['x_train'], data['y_train'],
        validation_data=(data['x_val'], data['y_val']),
        epochs=EPOCHS,
        shuffle=True,
        batch_size=config['batch_size'],
        callbacks=[es],
        verbose=0,
    )
    elapsed = time.time() - t0
    h = history.history
    val_acc = np.array(h['val_accuracy'], dtype=float)
    best_epoch = int(np.argmax(val_acc))

    # Evaluate on held-out test set (never seen during training/HPO)
    test_loss, test_acc = model.evaluate(data['x_test'], data['y_test'], verbose=0)

    out = {
        'seed':              seed,
        'n_epochs_trained':  len(val_acc),
        'best_epoch':        best_epoch + 1,
        'best_val_accuracy': float(val_acc[best_epoch]),
        'final_val_accuracy':float(val_acc[-1]),
        'test_accuracy':     float(test_acc),
        'test_loss':         float(test_loss),
        'history':           {
            'accuracy':     [float(v) for v in h['accuracy']],
            'val_accuracy': [float(v) for v in h['val_accuracy']],
            'loss':         [float(v) for v in h['loss']],
            'val_loss':     [float(v) for v in h['val_loss']],
        },
        'n_params':          int(model.count_params()),
        'elapsed_s':         float(elapsed),
    }
    keras.backend.clear_session()
    del model
    return out


In [ ]:
all_results = {}
out_dirs    = {}

for ds in DATASETS:
    print(f'\n############ {ds} ############')
    data = getDataset(ds)
    print('train:', data['x_train'].shape, '| test:', data['x_test'].shape)

    out_dir = RESULTS_ROOT / 'retrain' / ds
    out_dir.mkdir(parents=True, exist_ok=True)
    out_dirs[ds] = out_dir

    results = {}
    for algo, info in selected[ds].items():
        print(f'\n========== {ds} / {algo} ==========')
        print(f'  source : {info["tag"]}  (HPO val_acc = {info["val_accuracy"]:.4f})')
        runs = []
        for seed in SEEDS:
            print(f'  seed {seed} ...', end=' ', flush=True)
            r = train_one(info['config'], seed, data)
            runs.append(r)
            print(f'best_val_acc={r["best_val_accuracy"]:.4f}  '
                  f'test_acc={r["test_accuracy"]:.4f}  '
                  f'epochs={r["n_epochs_trained"]}  time={r["elapsed_s"]:.0f}s')
        results[algo] = {
            'source_run':       info['tag'],
            'source_dataset':   info['dataset'],
            'hpo_val_accuracy': info['val_accuracy'],
            'config':           info['config'],
            'runs':             runs,
        }

    all_results[ds] = results
    out_json = out_dir / 'retrain_results_bo.json'
    with open(out_json, 'w') as f:
        json.dump(results, f, indent=2)
    print(f'\nSaved: {out_json}')

    del data


## 4. Plot: validation accuracy per epoch, one line per algorithm

Each curve is the mean over the 3 seeds and the shaded band is +/-1 std.
Where a seed stopped earlier than the others, its history is padded with
`NaN` so that mean and std are computed only over the seeds that reached
that epoch.


In [ ]:
COLORS = {'GA': '#1f77b4', 'SADE': '#d62728', 'PSO': '#2ca02c',
          'GWO': '#9467bd', 'BO': '#ff7f0e'}


def pad_to(arr, n):
    out = np.full(n, np.nan, dtype=float)
    out[:len(arr)] = arr
    return out


def per_epoch_plot(ds, key, ylabel, fname, legend_loc):
    results = all_results[ds]
    max_len = max(r['n_epochs_trained'] for a in results.values() for r in a['runs'])
    epochs_axis = np.arange(1, max_len + 1)

    fig, ax = plt.subplots(figsize=(9.5, 5.6))
    for algo, payload in results.items():
        M = np.vstack([pad_to(np.array(r['history'][key]), max_len)
                       for r in payload['runs']])
        mean = np.nanmean(M, axis=0)
        std = np.nanstd(M, axis=0, ddof=1) if M.shape[0] > 1 else np.zeros_like(mean)
        c = COLORS.get(algo)
        ax.fill_between(epochs_axis, mean - std, mean + std, color=c, alpha=0.18)
        ax.plot(epochs_axis, mean, color=c, linewidth=2, marker='o', markersize=4,
                label=algo)

    ax.set_xlabel('Epoch')
    ax.set_ylabel(ylabel)
    ax.set_title(f'{ds}: final retraining, {ylabel.lower()} per epoch '
                 f'(mean +/- std over {len(SEEDS)} seeds)')
    ax.grid(True, alpha=0.3)
    ax.legend(loc=legend_loc, fontsize=10, framealpha=0.92)
    plt.tight_layout()

    for ext in ('png', 'pdf'):
        p = out_dirs[ds] / f'{fname}.{ext}'
        plt.savefig(p, dpi=160 if ext == 'png' else None, bbox_inches='tight')
        print('Saved:', p)
    plt.show()


for ds in DATASETS:
    per_epoch_plot(ds, 'val_accuracy', 'Validation accuracy',
                   'retrain_val_accuracy_per_epoch', 'lower right')


## 5. Validation loss per epoch

Useful to diagnose late overfit. Same format as the accuracy plot.


In [ ]:
for ds in DATASETS:
    per_epoch_plot(ds, 'val_loss', 'Validation loss',
                   'retrain_val_loss_per_epoch', 'upper right')


## 6. Summary table

One row per algorithm and dataset, with mean +/- std over the 3 seeds.


In [ ]:
for ds in DATASETS:
    rows = []
    for algo, payload in all_results[ds].items():
        val_accs  = np.array([r['best_val_accuracy'] for r in payload['runs']])
        test_accs = np.array([r['test_accuracy']     for r in payload['runs']])
        epochs    = np.array([r['best_epoch']        for r in payload['runs']])
        times     = np.array([r['elapsed_s']         for r in payload['runs']])
        nparam    = payload['runs'][0]['n_params']
        rows.append({
            'Algorithm':                 algo,
            'Best-run val_acc':          f"{payload['hpo_val_accuracy']:.4f}",
            'Val acc (mean+/-std)':      f"{val_accs.mean():.4f} +/- {val_accs.std(ddof=1):.4f}",
            'Test acc (mean+/-std)':     f"{test_accs.mean():.4f} +/- {test_accs.std(ddof=1):.4f}",
            'Best epoch (mean+/-std)':   f"{epochs.mean():.1f} +/- {epochs.std(ddof=1):.1f}",
            'Train time s (mean+/-std)': f"{times.mean():.1f} +/- {times.std(ddof=1):.1f}",
            '# Params':                  f"{nparam:,}",
            'Source run':                payload['source_run'],
            'Source dataset':            payload['source_dataset'],
        })

    df = pd.DataFrame(rows)
    print(f'=== {ds} ===')
    print(df.to_string(index=False))
    print()

    df.to_csv(out_dirs[ds] / 'retrain_summary.csv', index=False)
    md_lines = ['| ' + ' | '.join(df.columns) + ' |',
                '| ' + ' | '.join('---' for _ in df.columns) + ' |']
    for _, r in df.iterrows():
        md_lines.append('| ' + ' | '.join(str(r[c]) for c in df.columns) + ' |')
    (out_dirs[ds] / 'retrain_summary.md').write_text('\n'.join(md_lines), encoding='utf-8')
    print(f"Saved: {out_dirs[ds] / 'retrain_summary.csv'}")
    print(f"Saved: {out_dirs[ds] / 'retrain_summary.md'}\n")


## 7. Configurations used (for reproducibility)

The configuration retrained for each algorithm, with the search run it
came from so that a dataset mismatch is visible on sight.


In [ ]:
for ds in DATASETS:
    cfg_rows = []
    for algo, payload in all_results[ds].items():
        c = payload['config']
        cfg_rows.append({
            'Algorithm':     algo,
            'batch_size':    c['batch_size'],
            'base_filters':  c['base_filters'],
            'n_blocks':      c['n_blocks'],
            'dense_units':   c['dense_units'],
            'optimizer':     c['optimizer'],
            'pooling':       c['pooling'],
            'kernel_size':   c['kernel_size'],
            'learning_rate': c.get('learning_rate'),
            'dropout_rate':  c.get('dropout_rate'),
            'activation':    c.get('activation'),
            'source_run':    payload['source_run'],
            'hpo_val_acc':   f"{payload['hpo_val_accuracy']:.4f}",
        })

    df_cfg = pd.DataFrame(cfg_rows)
    print(f'=== {ds} ===')
    print(df_cfg.to_string(index=False))
    df_cfg.to_csv(out_dirs[ds] / 'retrain_configs.csv', index=False)
    print(f"Saved: {out_dirs[ds] / 'retrain_configs.csv'}\n")
